# Study 1 - Space

**Figure 1 (Combinatoric Space): Human vs Machine divergent thinking. Latest data (45-model midpoint machine + 12,147 human).**

- DAT scored with the Olson (2021) GloVe scorer (mean pairwise cosine distance x100).
- Palette: Human = purple (#5E348B), Machine = teal (#3CB7B0). Scatter outlined, darkened 20%, alpha 0.36.
- 1:1 plot aspect. Gray grids: Panel A horizontal only, Panel B both axes.

## Panel A - Divergent Thinking Score by Split
Bottom 10% / Middle 80% / Top 10%; n=500/group sampled; Welch t stars + mean diff (Machine-Human).

## Panel B - Cumulative Distinct Words
Top 10%, first 7 valid words; both sides capped at 500 sampled responses; between-respondent bootstrap 95% CI.


## Panel A code

In [ ]:
import csv, numpy as np, random
from scipy import stats
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
csv.field_size_limit(10**9); random.seed(42); np.random.seed(42)
def load(p,enc='utf-8-sig'):
    with open(p,newline='',encoding=enc,errors='replace') as f: return list(csv.DictReader(f))
H=load("/home/user/human_data_scored.csv"); M=load("/home/user/machine_data_merged.csv")
def sc(rows,k):
    o=[]
    for r in rows:
        try:o.append(float(r.get(k,'')))
        except:pass
    return np.array(o)
Hs=sc(H,'word_dat_score'); Ms=sc(M,'dat_score')
def splits(ss):
    lo,hi=np.percentile(ss,10),np.percentile(ss,90)
    return [ss[ss<=lo], ss[(ss>lo)&(ss<hi)], ss[ss>=hi]]
Hsp=splits(Hs); Msp=splits(Ms)
# SAMPLE: fixed n per group per side (not whole population)
SAMPLE_N=500
def samp(a): return np.random.choice(a, min(SAMPLE_N,len(a)), replace=False)
Hsp=[samp(a) for a in Hsp]; Msp=[samp(a) for a in Msp]
HUMAN="#5E348B"; MACHINE="#3CB7B0"
def darken(hexc,f=0.64):
    hexc=hexc.lstrip('#'); r,g,b=[int(hexc[i:i+2],16) for i in (0,2,4)]
    return (r*f/255,g*f/255,b*f/255)
HUMAN_D=darken(HUMAN); MACHINE_D=darken(MACHINE)
groups=["Bottom 10%","Middle 80%","Top 10%"]
def ci95(a): return 1.96*np.std(a,ddof=1)/np.sqrt(len(a))
def stars(p): return "***" if p<1e-3 else "**" if p<1e-2 else "*" if p<0.05 else "ns"
ANNOT_SIZE=10; ANNOT_COLOR="#333333"
fig,ax=plt.subplots(figsize=(7,7))
x=np.arange(3); w=0.36
for i in range(3):
    h=Hsp[i]; m=Msp[i]
    ax.bar(i-w/2, h.mean(), w, color=HUMAN, alpha=0.85, zorder=2)
    ax.bar(i+w/2, m.mean(), w, color=MACHINE, alpha=0.85, zorder=2)
    def jit(vals,center):
        return center+np.random.uniform(-w/2.6,w/2.6,len(vals)), vals
    jx,jv=jit(h,i-w/2); ax.scatter(jx,jv,s=12,facecolors='none',edgecolors=HUMAN_D,alpha=0.36,linewidths=0.6,zorder=3)
    jx,jv=jit(m,i+w/2); ax.scatter(jx,jv,s=12,facecolors='none',edgecolors=MACHINE_D,alpha=0.36,linewidths=0.6,zorder=3)
    ax.errorbar(i-w/2,h.mean(),yerr=ci95(h),color='black',capsize=4,lw=1.4,zorder=5)
    ax.errorbar(i+w/2,m.mean(),yerr=ci95(m),color='black',capsize=4,lw=1.4,zorder=5)
    t,p=stats.ttest_ind(h,m,equal_var=False)
    diff=m.mean()-h.mean()
    ytop=max(h.mean(),m.mean())+9
    ax.plot([i-w/2,i-w/2,i+w/2,i+w/2],[ytop-1.5,ytop,ytop,ytop-1.5],color='black',lw=1.1,zorder=5)
    ax.text(i,ytop+0.3,stars(p),ha='center',va='bottom',fontsize=ANNOT_SIZE,color=ANNOT_COLOR,zorder=6)
    ax.text(i,ytop+3.0,f"{diff:+.1f}",ha='center',va='bottom',fontsize=ANNOT_SIZE,weight='bold',color=ANNOT_COLOR,zorder=6)
    ax.text(i-w/2,ytop-3.2,f"{h.mean():.1f}",ha='center',va='top',fontsize=ANNOT_SIZE,weight='bold',color=ANNOT_COLOR,zorder=6)
    ax.text(i+w/2,ytop-3.2,f"{m.mean():.1f}",ha='center',va='top',fontsize=ANNOT_SIZE,weight='bold',color=ANNOT_COLOR,zorder=6)
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=11)
ax.set_ylabel("Divergent thinking score", fontsize=11); ax.set_ylim(60,105)
from matplotlib.patches import Patch
leg=ax.legend(handles=[Patch(color=HUMAN,label='Human'),Patch(color=MACHINE,label='Machine')], loc='upper left', fontsize=10)
leg._legend_box.align='left'
leg.set_title(None)
# sample-size note directly under the legend box, left-aligned with it
fig.canvas.draw()
bb=leg.get_window_extent().transformed(ax.transAxes.inverted())
ax.text(bb.x0+0.008, bb.y0-0.02, f'n={SAMPLE_N} per group,\nrandom sampled', transform=ax.transAxes, fontsize=8.5, color='#555', va='top', ha='left')
ax.set_title("Panel A - Divergent Thinking Score by Split", fontsize=12, weight='bold')

ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', color='#cccccc', linewidth=0.7, alpha=0.8, zorder=0)
ax.set_axisbelow(True)
# exact 1:1 aspect per Dawei's snippet
x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim()
ax.set_aspect(abs(x1-x0)/abs(y1-y0))
fig.tight_layout(); fig.savefig("/home/user/panelA.png",dpi=160,bbox_inches='tight'); plt.close(fig)
print("PANEL A v4 (sampled + 1:1) DONE")


## Panel B code

In [ ]:
import csv, pickle, numpy as np, random
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
csv.field_size_limit(10**9); random.seed(42); np.random.seed(42)
GLOVE=pickle.load(open("/home/user/repro/models/glove_validated.pickle","rb"))
def valid(w): 
    w=str(w).strip().lower(); return w if (w and w in GLOVE) else None
def load(p,enc='utf-8-sig'):
    with open(p,newline='',encoding=enc,errors='replace') as f: return list(csv.DictReader(f))
WCOLS=[f"word_{i}" for i in range(1,11)]
H=load("/home/user/human_data_scored.csv"); M=load("/home/user/machine_data_merged.csv")
def rws(rows,k):
    o=[]
    for r in rows:
        ws=[r[c] for c in WCOLS]; s=r.get(k,'')
        try:s=float(s)
        except:s=None
        o.append((ws,s))
    return o
Hd=[(w,s) for w,s in rws(H,'word_dat_score') if s is not None]
Md=[(w,s) for w,s in rws(M,'dat_score') if s is not None]
def top10(d):
    t=np.percentile([s for _,s in d],90); return [w for w,s in d if s>=t]
Ht=top10(Hd); Mt=top10(Md)
# first 7 valid words per response
def first7(ws):
    o=[];seen=set()
    for w in ws:
        v=valid(w)
        if v is None or v in seen: continue
        seen.add(v);o.append(v)
        if len(o)>=7:break
    return o
Ha=[first7(ws) for ws in Ht]; Ha=[x for x in Ha if len(x)==7]
Ma=[first7(ws) for ws in Mt]; Ma=[x for x in Ma if len(x)==7]
# rarefaction capped at SAME 500 sampled responses per side
CAP=500; STEP=5; REPS=40
def rarefy(pop):
    # Between-respondent BOOTSTRAP: at each k, resample respondents WITH REPLACEMENT
    # from the full pool, then accumulate distinct words. Band = spread across
    # respondent compositions (real heterogeneity), not just draw stability.
    N=len(pop); n=min(CAP,N); xs=list(range(STEP,n+1,STEP)); ys=[];lo=[];hi=[]
    BREPS=300
    for k in xs:
        c=[]
        for _ in range(BREPS):
            idx=np.random.randint(0,N,size=k)  # with replacement
            s=set()
            for i in idx: s.update(pop[i])
            c.append(len(s))
        ys.append(np.mean(c));lo.append(np.percentile(c,2.5));hi.append(np.percentile(c,97.5))
    return xs,ys,lo,hi
hx,hy,hl,hh=rarefy(Ha); mx,my,ml,mh=rarefy(Ma)
HUMAN="#5E348B"; MACHINE="#3CB7B0"
fig,ax=plt.subplots(figsize=(7,7))
ax.plot(hx,hy,color=HUMAN,lw=2,label="Human"); ax.fill_between(hx,hl,hh,color=HUMAN,alpha=0.2)
ax.plot(mx,my,color=MACHINE,lw=2,label="Machine"); ax.fill_between(mx,ml,mh,color=MACHINE,alpha=0.2)
ax.set_xlabel("Sampled responses", fontsize=11); ax.set_ylabel("Cumulative distinct words", fontsize=11)
ax.set_title("Panel B - Cumulative Distinct Words", fontsize=12, weight='bold')
from matplotlib.patches import Patch
leg=ax.legend(handles=[Patch(color=HUMAN,label='Human'),Patch(color=MACHINE,label='Machine')], loc='upper left', fontsize=10)
leg._legend_box.align='left'
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='both', color='#cccccc', linewidth=0.7, alpha=0.8, zorder=0)
ax.set_axisbelow(True)
fig.canvas.draw()
bb=leg.get_window_extent().transformed(ax.transAxes.inverted())
ax.text(bb.x0+0.008, bb.y0-0.02, "top 10%, first 7 words\n500 resp., respondent bootstrap 95% CI", transform=ax.transAxes, fontsize=8.5, color='#555', va='top', ha='left')
# endpoint distinct-word counts, top-center-right
ax.text(0.66,0.97,"Distinct words @500:",transform=ax.transAxes,fontsize=9.5,color="#333",va="top",ha="left",weight="bold")
ax.text(0.66,0.915,f"Human  {int(round(hy[-1]))}",transform=ax.transAxes,fontsize=10,color=HUMAN,va="top",ha="left",weight="bold")
ax.text(0.66,0.865,f"Machine  {int(round(my[-1]))}",transform=ax.transAxes,fontsize=10,color=MACHINE,va="top",ha="left",weight="bold")
ax.set_xlim(0,CAP+5)
x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim(); ax.set_aspect(abs(x1-x0)/abs(y1-y0))
fig.tight_layout(); fig.savefig("/home/user/panelB.png",dpi=160,bbox_inches='tight'); plt.close(fig)
print(f"Panel B done. human@500={hy[-1]:.0f} machine@500={my[-1]:.0f} (Ha={len(Ha)} Ma={len(Ma)})")


## Panel C - New Words per Sampled Response
Matches the paper's bottom-left panel (the DOWN-curve): marginal new distinct words added at each sampled response position (the slope of Panel B). Top 10%, first 7 words, n=500/group sampled, SEM band. Humans decay slowly (~3.3 new words at 500); machines saturate fast (~0.3).


In [ ]:
import csv, pickle, numpy as np, random
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
csv.field_size_limit(10**9); random.seed(42); np.random.seed(42)
GLOVE=pickle.load(open("/home/user/repro/models/glove_validated.pickle","rb"))
def valid(w):
    w=str(w).strip().lower(); return w if (w and w in GLOVE) else None
def load(p,enc='utf-8-sig'):
    with open(p,newline='',encoding=enc,errors='replace') as f: return list(csv.DictReader(f))
WCOLS=[f"word_{i}" for i in range(1,11)]
H=load("/home/user/human_data_scored.csv"); M=load("/home/user/machine_data_merged.csv")
def rws(rows,k):
    o=[]
    for r in rows:
        ws=[r[c] for c in WCOLS]; s=r.get(k,'')
        try:s=float(s)
        except:s=None
        o.append((ws,s))
    return o
Hd=[(w,s) for w,s in rws(H,'word_dat_score') if s is not None]
Md=[(w,s) for w,s in rws(M,'dat_score') if s is not None]
def top10(d):
    t=np.percentile([s for _,s in d],90); return [w for w,s in d if s>=t]
Ht=top10(Hd); Mt=top10(Md)
def first7(ws):
    o=[];seen=set()
    for w in ws:
        v=valid(w)
        if v is None or v in seen: continue
        seen.add(v);o.append(v)
        if len(o)>=7:break
    return o
Ha=[first7(ws) for ws in Ht if len(first7(ws))==7]
Ma=[first7(ws) for ws in Mt if len(first7(ws))==7]
# PANEL C (paper, bottom-left): NEW words per sampled response = marginal new distinct words
# at sample position k. Down-sloping: starts ~7, decays as vocabulary saturates.
CAP=500; REPS=200
def new_per(pop):
    pop=[set(a) for a in pop]
    N=len(pop); K=min(CAP,N)
    allreps=np.zeros((REPS,K))
    for r in range(REPS):
        order=np.random.permutation(N)[:K]  # random ordering, no replacement
        seen=set()
        for pos,i in enumerate(order):
            before=len(seen); seen|=pop[i]; allreps[r,pos]=len(seen)-before
    mean=allreps.mean(axis=0); sem=allreps.std(axis=0,ddof=1)/np.sqrt(REPS)
    xs=np.arange(1,K+1)
    return xs,mean,sem
hx,hy,hs=new_per(Ha); mx,my,ms=new_per(Ma)
HUMAN="#5E348B"; MACHINE="#3CB7B0"
fig,ax=plt.subplots(figsize=(7,7))
ax.plot(hx,hy,color=HUMAN,lw=1.6,label="Human"); ax.fill_between(hx,hy-1.96*hs,hy+1.96*hs,color=HUMAN,alpha=0.25)
ax.plot(mx,my,color=MACHINE,lw=1.6,label="Machine"); ax.fill_between(mx,my-1.96*ms,my+1.96*ms,color=MACHINE,alpha=0.25)
ax.set_xlabel("Sampled responses", fontsize=11); ax.set_ylabel("New distinct words added", fontsize=11)
ax.set_title("Panel C - New Words per Sampled Response", fontsize=12, weight='bold')
ax.text(0.60,0.95,"New words @500:",transform=ax.transAxes,fontsize=9.5,color="#333",va="top",ha="left",weight="bold")
ax.text(0.60,0.895,f"Human  {hy[-1]:.2f}",transform=ax.transAxes,fontsize=10,color=HUMAN,va="top",ha="left",weight="bold")
ax.text(0.60,0.84,f"Machine  {my[-1]:.2f}",transform=ax.transAxes,fontsize=10,color=MACHINE,va="top",ha="left",weight="bold")
from matplotlib.patches import Patch
leg=ax.legend(handles=[Patch(color=HUMAN,label='Human'),Patch(color=MACHINE,label='Machine')], loc='upper right', fontsize=10)
leg._legend_box.align='left'
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='both', color='#cccccc', linewidth=0.7, alpha=0.8, zorder=0); ax.set_axisbelow(True)
ax.text(0.98,0.66,f"n={CAP} per group,\nrandom sampled",transform=ax.transAxes,fontsize=8.5,color='#555',va='top',ha='right')
ax.set_xlim(0,CAP+5); ax.set_ylim(0,7.2)
x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim(); ax.set_aspect(abs(x1-x0)/abs(y1-y0))
fig.tight_layout(); fig.savefig("/home/user/panelC.png",dpi=160,bbox_inches='tight'); plt.close(fig)
print(f"Panel C new-words done. Human@500 {hy[-1]:.2f} Machine@500 {my[-1]:.2f}")


## Panel D - Categorical Diversity (dual-axis)
WordNet category diversity by hypernym depth level (1-8). LEFT axis: distinct category counts (Human purple, Machine teal). RIGHT axis: Human/Machine ratio (dashed orange). Top 10%, first 7 words, n=500/group sampled. Counts diverge sharply at fine levels (level 8: ~1,490 vs ~288); ratio rises 1.0x -> ~5.2x.


In [ ]:
import csv, pickle, numpy as np, random
from nltk.corpus import wordnet as wn
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
csv.field_size_limit(10**9); random.seed(42); np.random.seed(42)
GLOVE=pickle.load(open("/home/user/repro/models/glove_validated.pickle","rb"))
def valid(w):
    w=str(w).strip().lower(); return w if (w and w in GLOVE) else None
def load(p,enc='utf-8-sig'):
    with open(p,newline='',encoding=enc,errors='replace') as f: return list(csv.DictReader(f))
WCOLS=[f"word_{i}" for i in range(1,11)]
H=load("/home/user/human_data_scored.csv"); M=load("/home/user/machine_data_merged.csv")
def rws(rows,k):
    o=[]
    for r in rows:
        ws=[r[c] for c in WCOLS]; s=r.get(k,'')
        try:s=float(s)
        except:s=None
        o.append((ws,s))
    return o
Hd=[(w,s) for w,s in rws(H,'word_dat_score') if s is not None]
Md=[(w,s) for w,s in rws(M,'dat_score') if s is not None]
def top10(d):
    t=np.percentile([s for _,s in d],90); return [w for w,s in d if s>=t]
Ht=top10(Hd); Mt=top10(Md)
def first7(ws):
    o=[];seen=set()
    for w in ws:
        v=valid(w)
        if v is None or v in seen: continue
        seen.add(v);o.append(v)
        if len(o)>=7:break
    return o
Ha=[first7(ws) for ws in Ht if len(first7(ws))==7]
Ma=[first7(ws) for ws in Mt if len(first7(ws))==7]
SAMPLE_N=500
random.shuffle(Ha); random.shuffle(Ma); Ha=Ha[:SAMPLE_N]; Ma=Ma[:SAMPLE_N]
# WordNet category at a given DEPTH level along the hypernym path (root->...->word)
LEVELS=list(range(1,9))
def cats_at_level(words, level):
    cats=set()
    for w in words:
        ss=wn.synsets(w, pos=wn.NOUN)
        if not ss: continue
        paths=ss[0].hypernym_paths()
        if not paths: continue
        p=paths[0]  # root ... synset
        idx=min(level, len(p)-1)
        cats.add(p[idx].name())
    return cats
# pool words per group
Hwords=[w for a in Ha for w in a]; Mwords=[w for a in Ma for w in a]
ratio=[]; hcnt=[]; mcnt=[]
for L in LEVELS:
    hc=len(cats_at_level(Hwords,L)); mc=len(cats_at_level(Mwords,L))
    hcnt.append(hc); mcnt.append(mc); ratio.append(hc/mc if mc>0 else np.nan)
HUMAN="#5E348B"; MACHINE="#3CB7B0"; RATIO="#c0762e"
fig,ax=plt.subplots(figsize=(7,7))
# LEFT axis: distinct category COUNTS (two lines)
ax.plot(LEVELS,hcnt,'-o',color=HUMAN,lw=2,ms=6,zorder=3,label="Human (count)")
ax.plot(LEVELS,mcnt,'-o',color=MACHINE,lw=2,ms=6,zorder=3,label="Machine (count)")
ax.set_xlabel("WordNet category depth (level)", fontsize=11)
ax.set_ylabel("Distinct WordNet categories", fontsize=11)
ax.set_title("Panel D - Categorical Diversity", fontsize=12, weight='bold')
ax.spines[['top']].set_visible(False)
ax.grid(axis='both', color='#cccccc', linewidth=0.7, alpha=0.8, zorder=0); ax.set_axisbelow(True)
# RIGHT axis: ratio line
ax2=ax.twinx()
ax2.plot(LEVELS,ratio,'--s',color=RATIO,lw=1.8,ms=5,zorder=4,label="Ratio (H/M)")
ax2.axhline(1,color='gray',ls=':',lw=1)
for L,r in zip(LEVELS,ratio):
    ax2.annotate(f"{r:.1f}x",(L,r),textcoords="offset points",xytext=(0,7),ha='center',fontsize=8,color=RATIO)
ax2.set_ylabel("Ratio (Human / Machine)", fontsize=11, color=RATIO)
ax2.tick_params(axis='y', labelcolor=RATIO)
ax2.set_ylim(0,max([r for r in ratio if not np.isnan(r)])+1.2)
ax2.spines[['top']].set_visible(False)
# combined legend
from matplotlib.lines import Line2D
h=[Line2D([0],[0],color=HUMAN,marker='o',lw=2,label='Human (count)'),
   Line2D([0],[0],color=MACHINE,marker='o',lw=2,label='Machine (count)'),
   Line2D([0],[0],color=RATIO,marker='s',ls='--',lw=1.8,label='Ratio (H/M)')]
leg=ax.legend(handles=h, loc='upper left', fontsize=9); leg._legend_box.align='left'
ax.text(0.03,0.72,f"n={SAMPLE_N} per group,\nrandom sampled",transform=ax.transAxes,fontsize=8.5,color='#555',va='top',ha='left')
# 1:1 aspect on the primary axis box
x0,x1=ax.get_xlim(); y0,y1=ax.get_ylim(); ax.set_aspect(abs(x1-x0)/abs(y1-y0))
fig.tight_layout(); fig.savefig("/home/user/panelD.png",dpi=160,bbox_inches='tight'); plt.close(fig)
print("Panel D done. ratio by level:", [f"{r:.2f}" for r in ratio])
print("H cats:",hcnt,"M cats:",mcnt)


## Composite - Figure 1 (A/B/C/D)
Stitch the four saved panel PNGs into one 2x2 figure.


In [ ]:
# ---- Composite: stitch Panels A-D into one 2x2 Figure 1 ----
import matplotlib.pyplot as plt, matplotlib.image as mpimg
imgs=["panelA.png","panelB.png","panelC.png","panelD.png"]; labels=["A","B","C","D"]
fig,ax=plt.subplots(2,2,figsize=(17,17))
for a,img,lab in zip(ax.flat,imgs,labels):
    a.imshow(mpimg.imread(img)); a.axis('off')
    a.text(-0.02,1.02,lab,transform=a.transAxes,fontsize=22,weight='bold',va='top',ha='left')
fig.suptitle("Figure 1 - Combinatoric Space: Human vs Machine (latest data)", fontsize=20, weight='bold', y=0.995)
fig.tight_layout(rect=[0,0,1,0.985]); fig.savefig("figure1_composite.png",dpi=110,bbox_inches='tight')


## Panel D — Trial (waiting for Brian's comments)

**Status: exploratory, not the official Panel D.** The committed Panel D remains the line chart (Categorical Diversity by depth). This trial re-imagines it as a **shared radial dendrogram** of the WordNet hypernym paths — an alternative way to show the same categorical-diversity result.

What it shows:
- Root at center; each concept placed in an angular wedge under its parent (dendrogram layout).
- **All** hypernym paths used per word (longest path traced), so real branches and terminations appear — not just the first path.
- Edge width ∝ number of words traversing that link; leaf circle size/border ∝ ending-word frequency (per group).
- Human = purple (drawn under), Machine = teal (drawn on top).
- Radius warped so inner/middle/outer bands are in ratio ~1:2:1.5 (nuggets sit mid-tree).
- Depth rings numbered (grey); each group's **deepest ring** emphasized in its color (50% alpha) with a top label — Human reaches ring 18, Machine ring 15.
- n = 500 responses per group (top-10% by DAT, first 7 valid GloVe words).

Awaiting Brian's feedback before deciding whether this replaces or supplements the line-chart Panel D.

In [ ]:
import csv, pickle, numpy as np, random
from collections import Counter, defaultdict
from nltk.corpus import wordnet as wn
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
csv.field_size_limit(10**9); random.seed(42); np.random.seed(42)
GLOVE=pickle.load(open("/home/user/repro/models/glove_validated.pickle","rb"))
def valid(w):
    w=str(w).strip().lower(); return w if (w and w in GLOVE) else None
def load(p,enc='utf-8-sig'):
    with open(p,newline='',encoding=enc,errors='replace') as f: return list(csv.DictReader(f))
WCOLS=[f"word_{i}" for i in range(1,11)]
H=load("/home/user/human_data_scored.csv"); M=load("/home/user/machine_data_merged.csv")
def rws(rows,k):
    o=[]
    for r in rows:
        ws=[r[c] for c in WCOLS]; s=r.get(k,'')
        try:s=float(s)
        except:s=None
        o.append((ws,s))
    return o
Hd=[(w,s) for w,s in rws(H,'word_dat_score') if s is not None]
Md=[(w,s) for w,s in rws(M,'dat_score') if s is not None]
def top10(d):
    t=np.percentile([s for _,s in d],90); return [w for w,s in d if s>=t]
def first7(ws):
    o=[];seen=set()
    for w in ws:
        v=valid(w)
        if v is None or v in seen: continue
        seen.add(v);o.append(v)
        if len(o)>=7:break
    return o
Ha=[first7(ws) for ws in top10(Hd) if len(first7(ws))==7]
Ma=[first7(ws) for ws in top10(Md) if len(first7(ws))==7]
random.shuffle(Ha); random.shuffle(Ma)
Htokens=[w for a in Ha[:500] for w in a]; Mtokens=[w for a in Ma[:500] for w in a]

ROOT='entity.n.01'
def longest_path(w):
    ss=wn.synsets(w,pos=wn.NOUN)
    if not ss: return None
    ps=ss[0].hypernym_paths()
    if not ps: return None
    p=[s.name() for s in max(ps,key=len)]
    return p if p and p[0]==ROOT else None

# Build a SHARED tree (union of both groups) as parent->children, plus weighted edge counts per group + leaves
children=defaultdict(set); parent={}
edgeH=Counter(); edgeM=Counter(); leafH=Counter(); leafM=Counter()
def ingest(tokens,edge,leaf):
    for w in tokens:
        p=longest_path(w)
        if not p: continue
        for a,b in zip(p[:-1],p[1:]):
            children[a].add(b); parent[b]=a; edge[(a,b)]+=1
        leaf[p[-1]]+=1
ingest(Htokens,edgeH,leafH); ingest(Mtokens,edgeM,leafM)
# leaf counts of the SHARED tree for angular allocation
def n_leaves(node,memo={}):
    if node in memo: return memo[node]
    ch=children.get(node)
    if not ch: memo[node]=1; return 1
    memo[node]=sum(n_leaves(c) for c in ch); return memo[node]
# assign angular span to each node proportional to its leaf count (dendrogram layout)
ang={}; depth={}
def layout(node,a0,a1,d):
    depth[node]=d; ang[node]=(a0+a1)/2
    ch=sorted(children.get(node,[]))
    if not ch: return
    tot=sum(n_leaves(c) for c in ch); cur=a0
    for c in ch:
        span=(a1-a0)*n_leaves(c)/tot
        layout(c,cur,cur+span,d+1); cur+=span
layout(ROOT,0,2*np.pi,0)
maxd=max(depth.values())
def rad(d):
    if not maxd: return 0
    third=maxd/3.0
    b0=int(round(third)); b1=int(round(2*third))  # band edges in depth units
    # band radial spans in ratio 1 : 1.5 : 1
    s0,s1,s2=1.0,2.0,1.5; tot=s0+s1+s2
    R=maxd
    r0=R*s0/tot; r1=R*s1/tot; r2=R*s2/tot
    gaps=[0.0]*(maxd)
    n0=b0; n1=b1-b0; n2=maxd-b1
    for k in range(maxd):
        if k<b0: gaps[k]=r0/max(1,n0)
        elif k<b1: gaps[k]=r1/max(1,n1)
        else: gaps[k]=r2/max(1,n2)
    r=np.concatenate([[0],np.cumsum(gaps)])
    return r[d]
def pos(n): r=rad(depth[n]); return (r*np.cos(ang[n]), r*np.sin(ang[n]))
print(f"shared tree nodes {len(depth)}, max depth {maxd}; edgesH {len(edgeH)} edgesM {len(edgeM)}")

HUMAN="#5E348B"; MACHINE="#3CB7B0"
fig,ax=plt.subplots(figsize=(16,16))
for d in range(1,maxd+1):
    ax.add_patch(plt.Circle((0,0),rad(d),fill=False,color="#c9c9c9",lw=0.9,zorder=0))
def draw(edge,color,wmax):
    for (a,b),w in edge.items():
        if a in ang and b in ang:
            xa,ya=pos(a); xb,yb=pos(b)
            lw=1.3+12.0*(w/wmax)**1.3; al=0.35+0.6*(w/wmax)**0.7
            ax.plot([xa,xb],[ya,yb],color=color,lw=lw,alpha=min(al,0.75),zorder=2,solid_capstyle='round')
draw(edgeH,HUMAN,max(edgeH.values()))
draw(edgeM,MACHINE,max(edgeM.values()))  # green on top
PICKS=[]  # (name, word, color)
def leaves(leaf,color,n_inner=10,n_outer=26):
    lm=max(leaf.values())
    for n,w in leaf.items():
        if n in ang:
            x,y=pos(n); ax.scatter([x],[y],s=32+900*(w/lm)**1.4,facecolors='none',edgecolors=color,linewidths=1.3+4.5*(w/lm)**1.2,alpha=0.9,zorder=6)
    # INNER: max 3 per quadrant (by frequency)
    inner_by_q={0:[],1:[],2:[],3:[]}
    for n,w in leaf.items():
        if n in ang and depth[n]<10:
            q=int((ang[n]%(2*np.pi))//(np.pi/2))
            inner_by_q[q].append((n,w))
    inner_pick=[]
    for q in range(4):
        inner_pick+=sorted(inner_by_q[q],key=lambda kv:-kv[1])[:3]
    # OUTER: all candidates by frequency; overlap-skip happens at placement
    outer=sorted([(n,w) for n,w in leaf.items() if n in ang and depth[n]>=10],key=lambda kv:-kv[1])
    picks=inner_pick+outer
    for n,w in picks:
        word=n.split('.')[0].replace('_',' ')
        if 'man-of-war' in word: continue
        PICKS.append((n,word,color))
leaves(leafH,HUMAN); leaves(leafM,MACHINE)
# deterministic de-overlap: place each label at its node, but bump radius outward when
# angularly-close neighbors would collide. Sort by angle, walk, track last placed (ang,rad).
items=[]; seen_words=set()
for n,word,color in PICKS:
    if word in seen_words: continue   # drop duplicate label text
    seen_words.add(word)
    items.append((ang[n], rad(depth[n]), n, word, color))
items.sort()
placed=[]  # (angle, radius_used)
MINANG=0.20   # radians
STEP=1.6      # radial bump per collision
for a,r,n,word,color in items:
    ru=r
    # bump outward while too close to an already-placed label
    for pa,pr in placed:
        if abs(a-pa)<MINANG and abs(ru-pr)<1.8:
            ru=pr+STEP
    placed.append((a,ru))
    x,y=ru*np.cos(a), ru*np.sin(a)
ax.scatter([0],[0],s=40,color="#333",zorder=7)
# --- ring-number labels along the TOP wedge (fewer nodes up there) ---
for _d in range(1,maxd+1):
    _rr=rad(_d)
    ax.text(0.0, _rr, str(_d), fontsize=11, color="#c9c9c9", weight='bold', ha='center', va='center', zorder=7)
# --- deepest ring per group, in group color, thick + labeled ---
Hmax=max((depth[n] for n in leafH if n in depth), default=0)
Mmax=max((depth[n] for n in leafM if n in depth), default=0)
# offset each group's labels so human (outer) and machine (inner) never coincide.
for dmax,color,name,off in [(Hmax,HUMAN,"Human",0.0),(Mmax,MACHINE,"Machine",0.0)]:
    if dmax>0:
        rr=rad(dmax)
        ax.add_patch(plt.Circle((0,0),rr,fill=False,color=color,lw=4.0,alpha=0.5,zorder=4))
        # top label only
        x,y=rr*np.cos(np.pi/2), rr*np.sin(np.pi/2)
        ax.text(x,y,f"{name}: ring {dmax}",fontsize=15,weight='bold',color='white',ha='center',va='center',zorder=11,
                bbox=dict(boxstyle='round,pad=0.35',fc=color,ec='white',lw=1.4,alpha=1.0))
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0],color=HUMAN,lw=3,label='Human'),Line2D([0],[0],color=MACHINE,lw=3,label='Machine')],loc='upper left',fontsize=13)
ax.set_title(f"Panel D (experiment) — shared radial dendrogram of WordNet paths\n500 responses/group; edge width ~ #words on link; ending words circled; {maxd} depth rings",fontsize=14,weight='bold')
ax.set_xlim(-maxd-1,maxd+1); ax.set_ylim(-maxd-1,maxd+1); ax.axis('off'); ax.set_aspect('equal')
fig.tight_layout(); fig.savefig("/home/user/panelD_dendro.png",dpi=115,bbox_inches='tight'); plt.close(fig)
print("dendrogram done")
